# Gradient Boosting 시계열 예측

## Bike Sharing 수요의 1시간 선행 예측

시간순 평가조건을 유지하면서 Gradient Boosting과 직전 값 baseline을 같은 미래 구간에서 비교합니다.


## 이번 실습에서 완성할 것

- 1시간·24시간·168시간 lag의 시각 의미를 확인합니다.
- Boosting의 잔차 보정 과정을 작은 표로 읽습니다.
- 같은 미래행에서 직전 값 기준모형과 MAE·RMSE를 비교합니다.

**완료 결과:** 동일 미래행 비교표·그림, 개선량과 평가기간을 포함한 결과문장


## 이번 교시에서 사용할 데이터

### 강의 시연: Steel 15분·24시간 lag 자료

7교시와 같은 Steel 시계열에서 15분 전과 24시간 전 전력사용량을 feature로 만듭니다. 원자료는 35,040행이지만 lag가 존재하지 않는 초기·불연속 관측은 모형 입력에서 제외되므로 실제 학습행 수가 줄어듭니다.

- **target:** 현재 시점의 `Usage_kWh`
- **주요 예측정보:** 15분 전 사용량, 하루 전 같은 시각 사용량, 달력·부하조건
- **후보모형:** `HistGradientBoostingRegressor`
- **baseline:** 같은 미래기간의 persistence

### 적용 실습: Bike 시간별 수요

Bike 자료에서는 한 시간 전과 하루 전 수요를 이용해 같은 과거–미래 검증을 수행합니다. 날씨 열을 사용할 때는 예측시점에 실제 예보로 이용 가능한 정보인지 구분합니다. 수업 파일의 날씨 관측값을 사용한 결과를 실제 사전예측 성능으로 확대해서 말하지 않습니다.


## 실습에서 고정할 연구조건

| 그대로 유지할 방법 | Bike에서 새로 확인할 조건 |
|---|---|
| 시간순 80:20 분할과 persistence | 1·24·168시간 lag의 정확한 timestamp |
| 수업 전에 정한 Boosting 설정 | 결과시각 날씨정보를 쓴 조건부 benchmark 여부 |
| 같은 test의 MAE·RMSE 비교 | 다른 계절로 확대하기 전 필요한 검증 |

test 점수를 보고 학습률이나 tree 수를 다시 고르지 않습니다.


## 실제 실행 순서와 산출물

1. 1·24·168시간 lag가 가리키는 시각과 제외된 행 수를 확인합니다.
2. 고정된 설정으로 Boosting을 과거자료에 학습합니다.
3. 동일 미래행의 실제값·persistence·Boosting과 절대오차를 출력합니다.
4. MAE·RMSE 개선량, 평가기간과 입력정보 한계를 기록합니다.

**산출물:** 같은 미래행 비교표·세 선 그림, MAE·RMSE 표, 조건부 benchmark 문장


## 실행 환경


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행해 주세요."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 예측 시점에서 이용 가능한 변수

직전 값과 주기 lag를 timestamp join으로 만들고, calendar·날씨 정보의 가용 시점을 기록합니다.


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

raw = pd.read_csv(find_course_data("bike/hour.csv"), parse_dates=["dteday"])
raw["timestamp"] = raw["dteday"] + pd.to_timedelta(raw["hr"], unit="h")
raw = raw.sort_values("timestamp")
target_name = "cnt"
analysis_name = "Bike gradient boosting forecast"

def attach_lag(frame, hours, name):
    lag = frame[["timestamp", target_name]].copy()
    lag["timestamp"] = lag["timestamp"] + pd.Timedelta(hours=hours)
    return frame.merge(lag.rename(columns={target_name: name}), on="timestamp", how="left")

data = attach_lag(raw, 1, "lag_1h")
data = attach_lag(data, 24, "lag_24h")
data = attach_lag(data, 168, "lag_168h").dropna(subset=["lag_1h", "lag_24h", "lag_168h"]).copy()
data["hour_sin"] = np.sin(2*np.pi*data["hr"]/24)
data["hour_cos"] = np.cos(2*np.pi*data["hr"]/24)
features = ["lag_1h", "lag_24h", "lag_168h", "hour_sin", "hour_cos", "workingday", "temp", "hum", "windspeed"]
TARGET_TIME_WEATHER_COLUMNS = ['temp', 'hum', 'windspeed']
TARGET_TIME_WEATHER_USED = sorted(set(features) & set(TARGET_TIME_WEATHER_COLUMNS))
WEATHER_BENCHMARK_NOTE = '이 결과는 결과시각의 실제 날씨 관측값을 사용할 수 있다고 가정한 조건부 benchmark입니다. 실제 운영 예측에는 예보값과 예보오차 검증이 필요합니다.'
feature_availability = pd.DataFrame({
    '입력': ['lag_1h', 'workingday', *TARGET_TIME_WEATHER_COLUMNS],
    '값의 기준시각': ['1시간 전', '결과시각', '결과시각', '결과시각', '결과시각'],
    '예측 때 확보': ['직전 측정이 즉시 들어온다고 가정', '달력으로 사전 확인', '실측값만 제공', '실측값만 제공', '실측값만 제공'],
    '판정': ['가정 아래 사용', '사용 가능', '예보자료 필요', '예보자료 필요', '예보자료 필요'],
})
print('\n[입력정보의 실제 확보시점]')
display(feature_availability)
print(WEATHER_BENCHMARK_NOTE)
baseline_feature = "lag_1h"

data = data.sort_values("timestamp").reset_index(drop=True)
print("model rows:", len(data))
print("features:", features)
display(data[["timestamp", target_name, *features]].head(3))
print("\n[현재 수요와 세 개의 과거 수요]")
display(data[["timestamp", target_name, "lag_1h", "lag_24h", "lag_168h"]].head(8))
print("원자료 행 수:", len(raw), "| 세 lag가 모두 있는 모델 자료 행 수:", len(data))


## 2. 순차 보정과 시간순 학습

앞 80%로 학습하고 뒤 20%를 평가합니다. HistGradientBoosting은 작은 tree를 순차적으로 더해 남은 손실을 줄입니다.


In [ ]:
cut = int(len(data) * 0.80)
train, test = data.iloc[:cut], data.iloc[cut:]
X_train, y_train = train[features], train[target_name]
X_test, y_test = test[features], test[target_name]

model = HistGradientBoostingRegressor(
    learning_rate=0.06, max_iter=160, max_leaf_nodes=31,
    l2_regularization=0.1, random_state=42,
)
model.fit(X_train, y_train)
prediction = model.predict(X_test)
baseline = X_test[baseline_feature].to_numpy()
toy_boosting = pd.DataFrame({
    "실제값": [40.0, 60.0, 80.0],
    "첫 예측": [60.0, 60.0, 60.0],
})
toy_boosting["남은 잔차"] = toy_boosting["실제값"] - toy_boosting["첫 예측"]
toy_boosting["다음 tree 보정"] = 0.5 * toy_boosting["남은 잔차"]
toy_boosting["보정 후 예측"] = toy_boosting["첫 예측"] + toy_boosting["다음 tree 보정"]
print("\n[작은 숫자로 보는 boosting 한 단계]")
display(toy_boosting)

prediction_view = pd.DataFrame({
    "timestamp": test["timestamp"].iloc[:10].to_numpy(),
    "실제 수요": y_test.iloc[:10].to_numpy(),
    "직전 값 baseline": baseline[:10],
    "boosting 예측": prediction[:10],
})
print("\n[같은 미래 시각의 실제값·baseline·모형 예측]")
display(prediction_view.round(2))


## 3. Persistence 대비 예측 성능

같은 미래 test 기간에서 persistence와 boosting의 MAE·RMSE를 비교하고 첫 구간의 실제–예측 흐름을 살펴봅니다.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

result_table = pd.DataFrame([
    {"model": "persistence", "MAE": mean_absolute_error(y_test, baseline), "RMSE": root_mean_squared_error(y_test, baseline)},
    {"model": "hist gradient boosting", "MAE": mean_absolute_error(y_test, prediction), "RMSE": root_mean_squared_error(y_test, prediction)},
])
result_by_model = result_table.set_index("model")
baseline_mae = float(result_by_model.loc["persistence", "MAE"])
candidate_mae = float(result_by_model.loc["hist gradient boosting", "MAE"])
baseline_rmse = float(result_by_model.loc["persistence", "RMSE"])
candidate_rmse = float(result_by_model.loc["hist gradient boosting", "RMSE"])
mae_difference = baseline_mae - candidate_mae
rmse_difference = baseline_rmse - candidate_rmse
mae_direction = "낮았습니다" if mae_difference >= 0 else "높았습니다"
rmse_direction = "낮았습니다" if rmse_difference >= 0 else "높았습니다"
evaluation_period = f"{test['timestamp'].min():%Y-%m-%d %H:%M}–{test['timestamp'].max():%Y-%m-%d %H:%M}"
metric_unit = '건/시간'
data_scope = 'Bike 시간당 대여자료'
interpretation_scope = (
    WEATHER_BENCHMARK_NOTE
    if True and "WEATHER_BENCHMARK_NOTE" in globals() and TARGET_TIME_WEATHER_USED
    else "한 번의 시간순 평가기간 결과이며 다른 기간의 성능은 추가 검증이 필요합니다."
)
result_claim = (
    f"{data_scope}의 {evaluation_period} 시간순 test에서 HistGradientBoosting의 MAE는 "
    f"{candidate_mae:.2f} {metric_unit}였으며, persistence의 {baseline_mae:.2f} {metric_unit}보다 "
    f"{abs(mae_difference):.2f} {metric_unit} {mae_direction}. "
    f"RMSE는 {candidate_rmse:.2f} {metric_unit}였으며, persistence의 {baseline_rmse:.2f} {metric_unit}보다 "
    f"{abs(rmse_difference):.2f} {metric_unit} {rmse_direction}. {interpretation_scope}"
)
print("\n[평가자료에서 자동으로 만든 결과문장]")
print(result_claim)

display(result_table)

view = min(200, len(test))
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(test["timestamp"].iloc[:view], y_test.iloc[:view], label="observed", linewidth=1.8)
ax.plot(test["timestamp"].iloc[:view], baseline[:view], label="persistence", linewidth=1.3)
ax.plot(test["timestamp"].iloc[:view], prediction[:view], label="boosting", linewidth=1.3)
ax.set(title="Chronological test: observed vs prediction", xlabel="Time", ylabel=target_name)
ax.legend()
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()
error_view = pd.DataFrame({
    "timestamp": test["timestamp"].iloc[:8].to_numpy(),
    "실제 수요": y_test.iloc[:8].to_numpy(),
    "직전 값 baseline": baseline[:8],
    "boosting 예측": prediction[:8],
})
error_view["baseline 절대오차"] = (error_view["실제 수요"] - error_view["직전 값 baseline"]).abs()
error_view["boosting 절대오차"] = (error_view["실제 수요"] - error_view["boosting 예측"]).abs()
print("\n[같은 미래 행의 절대오차]")
display(error_view.round(2))
print("[같은 미래 test 전체의 MAE·RMSE]")
display(result_table.round(3))


## 4. Boosting 결과 검증

시간 순서, feature 결측, baseline 비교, 유한 예측을 확인합니다.


In [ ]:
checks = pd.Series({
    "weather benchmark labeled": (not TARGET_TIME_WEATHER_USED) or ("조건부 benchmark" in WEATHER_BENCHMARK_NOTE),
    "result claim contains evaluated evidence": evaluation_period in result_claim and metric_unit in result_claim and all(f"{value:.2f}" in result_claim for value in [baseline_mae, candidate_mae, baseline_rmse, candidate_rmse]),
    "result scope recorded": len(interpretation_scope.strip()) >= 30,
    "strict time order": train["timestamp"].max() < test["timestamp"].min(),
    "no feature missing": not data[features].isna().any().any(),
    "finite prediction": np.isfinite(prediction).all(),
    "same test length": len(prediction) == len(y_test),
    "baseline comparison complete": set(result_table["model"]) == {"persistence", "hist gradient boosting"},
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("시계열 예측 비교 검증: PASS")


## 연구 기록

다음 내용을 결과표와 시각 경계에서 확인해 작성합니다.

1. **고정한 모형 설정과 test를 설정 선택에 사용하지 않은 근거:**
2. **같은 미래기간의 persistence와 Boosting MAE·RMSE:**
3. **baseline 대비 개선량과 단위:**
4. **결과시각 날씨 관측을 사용한 조건부 benchmark 여부:**
5. **다음에 추가할 계절 또는 여러 예측 기준시점 검증:**

### 선택 검토: ChatGPT에 물어볼 때

```text
같은 미래기간의 기준모형·Boosting MAE와 RMSE, 기간, 입력정보 가용성 표를 아래에 제공합니다. 비교조건의 일치 여부와 결과문장의 범위를 점검하고, train 내부에서 검토할 설정 하나와 추가 시간검증 하나만 제시해 주세요.
```


## 완료 확인

마지막 출력이 `시계열 예측 비교 검증: PASS`인지 확인해 주세요. 이 검사는 필요한 계산이 실행됐는지 확인할 뿐, 연구 주장의 타당성을 자동 판정하지 않습니다.


## 결과를 정리할 질문

- Persistence와 Boosting은 같은 미래기간·입력정보·MAE·RMSE로 비교됐습니까?
- 고정한 설정에서 후보의 오차는 기준모형보다 얼마나 달랐으며 단위는 무엇입니까?
- 설정을 바꾸려면 최종 test가 아니라 train 내부의 어떤 검증절차가 필요합니까?
